# Vision Transformer (ViT) Baseline - 500 Samples

This notebook trains a Vision Transformer (ViT-B/16) model for document type classification using 500 samples from the dataset.

**Model Architecture:** Vision Transformer (ViT-B/16) with ImageNet pre-training  
**Dataset Size:** 500 images (balanced across 3 classes)  
**Training Configuration:**
- Batch size: 32
- Learning rate: 2e-5
- Epochs: 30
- Optimizer: Adam with weight decay 1e-4
- Dropout: 0.3


## Data Loading and Splitting

Load the dataset CSV file, sample 500 images (if needed), and split into train/validation/test sets with stratified sampling to maintain class balance.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# קבע כאן כמה תמונות אתה רוצה בסך הכל (None = כל הדאטה סט)
SAMPLE_SIZE = 500  # שנה את המספר הזה לפי הצורך

df = pd.read_csv("/Users/roy-siftt/final-project/datasets/idnet/document_type_classification_country_split/data/dataset.csv")

print(f"📊 גודל דאטה סט מקורי: {len(df)} תמונות")
print(f"\nהתפלגות במקור:")
print(df['label'].value_counts().sort_index())

# אם רוצים דאטה סט קטן יותר, דגום באופן מייצג
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df):
    df = df.groupby('label', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), int(SAMPLE_SIZE * len(x) / len(df))), random_state=42)
    ).reset_index(drop=True)
    print(f"\n📊 גודל דאטה סט לאחר דגימה: {len(df)} תמונות")
    print(f"\nהתפלגות לאחר דגימה:")
    print(df['label'].value_counts().sort_index())

# חלוקה ל-train/test/val
train_df, test_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df['label'], random_state=42)

print(f"\n{'='*60}")
print(f"📈 חלוקה סופית:")
print(f"{'='*60}")

print(f"\n🎓 Train Set: {len(train_df)} תמונות")
print(train_df['label'].value_counts().sort_index())
print(f"   סה\"כ: {train_df['label'].value_counts().sum()}")

print(f"\n✅ Validation Set: {len(val_df)} תמונות")
print(val_df['label'].value_counts().sort_index())
print(f"   סה\"כ: {val_df['label'].value_counts().sum()}")

print(f"\n🎯 Test Set: {len(test_df)} תמונות")
print(test_df['label'].value_counts().sort_index())
print(f"   סה\"כ: {test_df['label'].value_counts().sum()}")

print(f"\n{'='*60}")
print(f"💾 שומר קבצים...")

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

print(f"✅ הקבצים נשמרו בהצלחה!")
print(f"{'='*60}")


📊 גודל דאטה סט מקורי: 3600 תמונות

התפלגות במקור:
label
driver_license    1200
id_card           1200
passport          1200
Name: count, dtype: int64

📊 גודל דאטה סט לאחר דגימה: 498 תמונות

התפלגות לאחר דגימה:
label
driver_license    166
id_card           166
passport          166
Name: count, dtype: int64

📈 חלוקה סופית:

🎓 Train Set: 359 תמונות
label
driver_license    120
id_card           120
passport          119
Name: count, dtype: int64
   סה"כ: 359

✅ Validation Set: 64 תמונות
label
driver_license    21
id_card           21
passport          22
Name: count, dtype: int64
   סה"כ: 64

🎯 Test Set: 75 תמונות
label
driver_license    25
id_card           25
passport          25
Name: count, dtype: int64
   סה"כ: 75

💾 שומר קבצים...
✅ הקבצים נשמרו בהצלחה!


/var/folders/7v/16zlk2l53qq1v3dqsttk28zh0000gn/T/ipykernel_59393/71616216.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('label', group_keys=False).apply(


In [2]:
from torch.utils.data import Dataset
from PIL import Image
import torch

class DocumentDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

        # Normalize labels (lowercase + strip spaces)
        self.df['label'] = self.df['label'].str.lower().str.strip()

        # Label mapping
        self.label_map = {
            "id_card": 0,
            "passport": 1,
            "driver_license": 2
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert("RGB")

        label_str = row['label']

        if label_str not in self.label_map:
            raise ValueError(f"Label '{label_str}' not found in label_map!")

        label = self.label_map[label_str]

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label)


In [3]:
from torchvision import transforms

# Transform עם Data Augmentation - מותאם ל-ViT
# ViT עובד טוב עם 224x224 (כמו ResNet)
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),              # חיתוך אקראי
    transforms.RandomHorizontalFlip(p=0.3),  # היפוך אופקי
    transforms.RandomRotation(15),           # סיבוב אקראי
    transforms.ColorJitter(                  # שינויי צבע/בהירות
        brightness=0.3,
        contrast=0.3,
        saturation=0.2,
        hue=0.1
    ),
    transforms.RandomAffine(                 # טרנספורמציות גיאומטריות
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225]),  # ImageNet normalization (ViT pre-trained על ImageNet)
    transforms.RandomErasing(p=0.3)          # מחיקת אזורים אקראיים
])

# Transform לvalidation/test - ללא augmentation
val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])


In [4]:
from torch.utils.data import DataLoader

# שימוש ב-transforms שונים לtrain ול-val/test
train_dataset = DocumentDataset(pd.read_csv("train.csv"), train_transform)
val_dataset   = DocumentDataset(pd.read_csv("val.csv"), val_test_transform)
test_dataset  = DocumentDataset(pd.read_csv("test.csv"), val_test_transform)

# ViT עם דאטה סט קטן - batch size קטן יותר (32)
# הערה: num_workers=0 ב-MPS כדי למנוע בעיות
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"📊 Dataset sizes:")
print(f"   Train: {len(train_dataset)}")
print(f"   Val:   {len(val_dataset)}")
print(f"   Test:  {len(test_dataset)}")


📊 Dataset sizes:
   Train: 359
   Val:   64
   Test:  75


In [5]:
import torch
import torch.nn as nn
from torchvision import models

class ViTClassifier(nn.Module):
    def __init__(self, num_classes=3, pretrained=True, dropout=0.5):
        super(ViTClassifier, self).__init__()
        
        # טוען Vision Transformer pre-trained על ImageNet
        if pretrained:
            # ViT-Base/16 - גרסה מאוזנת בין ביצועים לגודל
            self.vit = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        else:
            self.vit = models.vit_b_16(weights=None)
        
        # משנה את השכבה האחרונה + מוסיף Dropout לregularization
        num_features = self.vit.heads.head.in_features
        self.vit.heads.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, num_classes)
        )
        
    def forward(self, x):
        return self.vit(x)


In [6]:
import matplotlib.pyplot as plt
import numpy as np
import torchvision

def show_batch(loader):
    imgs, labels = next(iter(loader))
    grid = torchvision.utils.make_grid(imgs, nrow=8, normalize=True)
    plt.figure(figsize=(12,8))
    plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
    plt.axis('off')

show_batch(train_loader)


KeyboardInterrupt: 

In [ ]:
for i in range(10):
    print("PATH:", train_dataset.df.iloc[i]['image_path'])
    print("LABEL:", train_dataset.df.iloc[i]['label'])
    img, lbl = train_dataset[i]
    print("LOADER LABEL:", lbl)
    print("----")


PATH: images/01391.png
LABEL: id_card
LOADER LABEL: tensor(0)
----
PATH: images/03037.png
LABEL: passport
LOADER LABEL: tensor(1)
----
PATH: images/04794.png
LABEL: driver_license
LOADER LABEL: tensor(2)
----
PATH: images/04116.png
LABEL: id_card
LOADER LABEL: tensor(0)
----
PATH: images/03839.png
LABEL: id_card
LOADER LABEL: tensor(0)
----
PATH: images/03746.png
LABEL: id_card
LOADER LABEL: tensor(0)
----
PATH: images/03631.png
LABEL: id_card
LOADER LABEL: tensor(0)
----
PATH: images/01908.png
LABEL: driver_license
LOADER LABEL: tensor(2)
----
PATH: images/03926.png
LABEL: id_card
LOADER LABEL: tensor(0)
----
PATH: images/02145.png
LABEL: driver_license
LOADER LABEL: tensor(2)
----


In [ ]:
import torch
from torch.optim import Adam
import torch.nn as nn
from tqdm import tqdm   # בר אלכטר יפה
import numpy as np

# בחירת מכשיר: CUDA (NVIDIA GPU) -> MPS (Apple Silicon GPU) -> CPU
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon GPU (M1/M2/M3)
else:
    device = "cpu"

# 🚀 יצירת מודל Vision Transformer עם pre-training!
# pretrained=True -> משתמש בידע מוקדם מ-ImageNet (זה אמור להיות הרבה יותר טוב!)
model = ViTClassifier(num_classes=3, pretrained=True, dropout=0.3).to(device)

criterion = nn.CrossEntropyLoss()

# Learning rate מותאם ל-ViT pre-trained עם דאטה סט קטן (500 תמונות)
# Learning rate גבוה יותר לדאטה סט קטן
optimizer = Adam(model.parameters(), lr=2e-5, weight_decay=1e-4)

print(f"Using device: {device}")
print(f"Model: Vision Transformer (ViT-B/16) - WITH ImageNet Pre-training! 🎯")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Learning rate: {optimizer.param_groups[0]['lr']}")
print(f"Dropout: 0.3")
print(f"Batch size: 32")
print("\n🚀 ViT עם pre-training אמור ללמוד הרבה יותר מהר ולהיות מדויק יותר!\n")

def evaluate(loader, verbose=False):
    model.eval()
    total = 0
    correct = 0
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = correct / total
    avg_loss = total_loss / len(loader)
    
    if verbose:
        # בדיקה מפורטת - מה המודל חוזה?
        print(f"\nSample predictions (first 20):")
        for i in range(min(20, len(all_preds))):
            print(f"  True: {all_labels[i]}, Predicted: {all_preds[i]}, {'✓' if all_preds[i] == all_labels[i] else '✗'}")
        
        # Confusion matrix
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(all_labels, all_preds)
        print(f"\nConfusion Matrix:")
        print(cm)
    
    return accuracy, avg_loss

# בדיקה ראשונית לפני אימון
print("\n" + "="*60)
print("=== Initial Evaluation (Before Training) ===")
print("="*60)
initial_train_acc, initial_train_loss = evaluate(train_loader, verbose=True)
initial_val_acc, initial_val_loss = evaluate(val_loader)
print(f"\n📊 Train Acc (before): {initial_train_acc:.4f}")
print(f"📊 Val Acc (before):   {initial_val_acc:.4f}")
print("="*60)

# מעקב אחר התהליך - נשמור את הביצועים בכל epoch
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

num_epochs = 30  # ViT עם pre-training אמור להתכנס מהר יותר

for epoch in range(num_epochs):
    model.train()
    running_loss = 0

    print(f"\n=== Epoch {epoch+1} ===")

    # tqdm ל progress bar יפה
    for imgs, labels in tqdm(train_loader, desc="Training"):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # חישובי validation
    train_accuracy, train_loss = evaluate(train_loader)
    val_accuracy, val_loss = evaluate(val_loader)
    
    # שמירה להיסטוריה
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_accuracy)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_accuracy)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_accuracy:.4f}")

print("\n" + "="*60)
print("🎯 FINAL EVALUATION ON TEST SET (Never seen during training)")
print("="*60)
test_accuracy, test_loss = evaluate(test_loader, verbose=True)
print(f"\n📊 Final Test Accuracy: {test_accuracy:.4f}")
print(f"📊 Final Test Loss: {test_loss:.4f}")

# 📊 הצגת גרפים של תהליך הלמידה
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# גרף 1: Loss לאורך הזמן
axes[0].plot(history['train_loss'], label='Train Loss', marker='o', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss During Training', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# גרף 2: Accuracy לאורך הזמן
axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Accuracy', marker='s', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Accuracy During Training', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("📈 Training Summary:")
print("="*60)
print(f"🎯 Best Train Accuracy: {max(history['train_acc']):.4f} (Epoch {history['train_acc'].index(max(history['train_acc']))+1})")
print(f"🎯 Best Val Accuracy:   {max(history['val_acc']):.4f} (Epoch {history['val_acc'].index(max(history['val_acc']))+1})")
print(f"🎯 Final Test Accuracy: {test_accuracy:.4f}")
print(f"\n💡 ViT עם pre-training אמור להציג ביצועים מעולים!")


Using device: mps
Model: Vision Transformer (ViT-B/16) - WITH ImageNet Pre-training! 🎯
Number of parameters: 85,800,963
Learning rate: 2e-05
Dropout: 0.3
Batch size: 32

🚀 ViT עם pre-training אמור ללמוד הרבה יותר מהר ולהיות מדויק יותר!


=== Initial Evaluation (Before Training) ===

Sample predictions (first 20):
  True: 2, Predicted: 2, ✓
  True: 1, Predicted: 2, ✗
  True: 2, Predicted: 0, ✗
  True: 0, Predicted: 1, ✗
  True: 0, Predicted: 1, ✗
  True: 2, Predicted: 2, ✓
  True: 1, Predicted: 1, ✓
  True: 2, Predicted: 2, ✓
  True: 2, Predicted: 1, ✗
  True: 1, Predicted: 1, ✓
  True: 0, Predicted: 2, ✗
  True: 1, Predicted: 2, ✗
  True: 1, Predicted: 1, ✓
  True: 2, Predicted: 2, ✓
  True: 1, Predicted: 1, ✓
  True: 2, Predicted: 2, ✓
  True: 1, Predicted: 1, ✓
  True: 0, Predicted: 2, ✗
  True: 1, Predicted: 1, ✓
  True: 0, Predicted: 2, ✗

Confusion Matrix:
[[14 69 37]
 [ 5 92 22]
 [ 1 48 71]]

📊 Train Acc (before): 0.4930
📊 Val Acc (before):   0.5000

=== Epoch 1 ===


Training: 100%|██████████| 12/12 [00:17<00:00,  1.47s/it]


Train Loss: 0.1825 | Train Acc: 0.9944
Val   Loss: 0.1057 | Val   Acc: 1.0000

=== Epoch 2 ===


Training:  75%|███████▌  | 9/12 [00:11<00:03,  1.33s/it]


KeyboardInterrupt: 